# Unified Grokking Framework — Demo Notebook

This notebook demonstrates the **grokking_unifier** framework which unifies
concepts from five grokking repositories:

| Repo | Key Contribution |
|------|------------------|
| `7vik/grokking` | Clean modular-arithmetic + Transformer setup |
| `neelnanda-io/Grokking` | Mechanistic interpretability, 1-layer Transformer |
| `ironjr/grokfast` | Gradient filtering to accelerate grokking |
| `nmallinar/rfm-grokking` | MLP / kernel baselines, weight-norm analysis |
| `Tikquuss/grokking_fda` | Activation / FDA-style diagnostics |

## What this notebook covers
1. **Baseline Transformer** — classic slow grokking on modular addition
2. **Grokfast-EMA Transformer** — accelerated grokking via gradient filtering
3. **MLP baseline** — comparison with a different architecture
4. **Side-by-side comparison** — accuracy curves, loss curves, and weight norms
5. **Extensibility demo** — how to add your own custom metric / hypothesis

## 0. Setup

**Running on Google Colab?**  
Upload the entire `grokking_unifier/` folder to Colab's working directory  
(or upload a zip and unzip it). The cell below handles path detection automatically.

In [ ]:
import sys, os

# --- Colab detection & path setup ---
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    # If running on Colab, ensure grokking_unifier is in the working dir
    # Upload the grokking_unifier/ folder to /content/ or mount Google Drive
    if not os.path.isdir('/content/grokking_unifier'):
        print('⚠️  Please upload the grokking_unifier/ folder to /content/')
        print('    Or mount your Drive and adjust the path below.')
    sys.path.insert(0, '/content')
else:
    # Local: add current dir and parent dir to path
    sys.path.insert(0, os.getcwd())
    sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

import torch
import numpy as np
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
import matplotlib.pyplot as plt

from grokking_unifier import (
    ModularArithmeticDataset,
    create_model,
    get_available_models,
    get_available_operations,
    get_available_metrics,
    Trainer,
    GrokfastEMA,
    WeightNormCallback,
    GradientNormCallback,
    register_operation,
    register_metric,
    MetricCallback,
)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"PyTorch  : {torch.__version__}")
print(f"Device   : {DEVICE}")
print(f"Models   : {get_available_models()}")
print(f"Operations: {get_available_operations()}")
print(f"Metrics  : {get_available_metrics()}")

## 1. Create Dataset

We use **modular addition mod 97** with a **30% training split** — the
canonical setup from Power et al. (2022) and Neel Nanda's grokking work.

We use **full-batch** training (all training samples in one batch) which is
standard in grokking experiments and ~2x faster than mini-batching.

In [ ]:
# ===== HYPERPARAMETERS (edit these) =====
PRIME        = 97
TRAIN_FRAC   = 0.3
BATCH_SIZE   = 4096   # full-batch (all 2823 train samples fit in one batch)
SEED         = 42
EPOCHS       = 8000   # 8000 epochs (~45s on Colab GPU) for full grokking transition
LR           = 1e-3
WEIGHT_DECAY = 1.0
LOG_INTERVAL = 1000
# ========================================

dataset = ModularArithmeticDataset(
    operation="addition",
    p=PRIME,
    train_fraction=TRAIN_FRAC,
    seed=SEED,
)

train_loader = dataset.get_train_loader(batch_size=BATCH_SIZE)
val_loader   = dataset.get_val_loader(batch_size=BATCH_SIZE)

print(dataset)
print(f"Dataset summary: {dataset.summary()}")

## 2. Experiment 1 — Baseline Transformer (no Grokfast)

Classic grokking setup:
- 1-layer Transformer, d_model=128, 4 heads, d_mlp=512
- AdamW with **lr=1e-3, weight_decay=1.0** (strong regularisation)
- The network first memorises (train acc → 1.0), then *much later*
  generalises (val acc → 1.0) — that delayed generalisation is **grokking**.

In [ ]:
torch.manual_seed(SEED)
baseline_model = create_model(
    "transformer", p=PRIME, d_model=128, n_heads=4, d_mlp=512, n_layers=1
)
baseline_opt = torch.optim.AdamW(
    baseline_model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY, betas=(0.9, 0.98)
)
baseline_wn = WeightNormCallback()
baseline_gn = GradientNormCallback()

baseline_trainer = Trainer(
    model=baseline_model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=baseline_opt,
    callbacks=[baseline_wn, baseline_gn],
    device=DEVICE,
    log_interval=LOG_INTERVAL,
)

print("Training BASELINE Transformer...")
baseline_hist = baseline_trainer.train(epochs=EPOCHS)

## 3. Experiment 2 — Grokfast-EMA Transformer

Same architecture and hyperparameters, but with the **Grokfast-EMA** gradient
filter (α=0.98, λ=2.0) from Lee et al.  This amplifies slow gradient
components to make generalisation happen *significantly* earlier.

In [ ]:
torch.manual_seed(SEED)
gf_model = create_model(
    "transformer", p=PRIME, d_model=128, n_heads=4, d_mlp=512, n_layers=1
)
gf_opt = torch.optim.AdamW(
    gf_model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY, betas=(0.9, 0.98)
)
gf_wn = WeightNormCallback()
gf_gn = GradientNormCallback()
gf_filter = GrokfastEMA(gf_model, alpha=0.98, lamb=2.0)

gf_trainer = Trainer(
    model=gf_model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=gf_opt,
    callbacks=[gf_wn, gf_gn],
    grokfast=gf_filter,
    device=DEVICE,
    log_interval=LOG_INTERVAL,
)

print("Training GROKFAST-EMA Transformer...")
gf_hist = gf_trainer.train(epochs=EPOCHS)

## 4. Experiment 3 — MLP Baseline

An MLP baseline (one-hot input, 2 hidden layers, 256 units) for comparison
with a fundamentally different architecture.

In [ ]:
torch.manual_seed(SEED)
mlp_model = create_model("mlp", p=PRIME, hidden_dim=256, n_layers=2)
mlp_opt = torch.optim.AdamW(
    mlp_model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY, betas=(0.9, 0.98)
)
mlp_wn = WeightNormCallback()

mlp_trainer = Trainer(
    model=mlp_model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=mlp_opt,
    callbacks=[mlp_wn],
    device=DEVICE,
    log_interval=LOG_INTERVAL,
)

print("Training MLP baseline...")
mlp_hist = mlp_trainer.train(epochs=EPOCHS)

## 5. Results Comparison

### 5a. Training & Validation Accuracy

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

configs = [
    ('Baseline Transformer', baseline_hist, 'tab:blue'),
    ('Grokfast-EMA Transformer', gf_hist, 'tab:orange'),
    ('MLP', mlp_hist, 'tab:green'),
]

for label, h, c in configs:
    axes[0].plot(h['epoch'], h['train_acc'], label=label, color=c, alpha=0.8)
    axes[1].plot(h['epoch'], h['val_acc'],   label=label, color=c, alpha=0.8)

axes[0].set(xlabel='Epoch', ylabel='Accuracy', title='Training Accuracy')
axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].set(xlabel='Epoch', ylabel='Accuracy',
            title='Validation Accuracy (Grokking = delayed rise)')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('accuracy_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: accuracy_comparison.png')

### 5b. Training & Validation Loss (log scale)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

for label, h, c in configs:
    axes[0].semilogy(h['epoch'], h['train_loss'], label=label, color=c, alpha=0.8)
    axes[1].semilogy(h['epoch'], h['val_loss'],   label=label, color=c, alpha=0.8)

axes[0].set(xlabel='Epoch', ylabel='Loss (log)', title='Training Loss')
axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].set(xlabel='Epoch', ylabel='Loss (log)', title='Validation Loss')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('loss_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: loss_comparison.png')

### 5c. Weight Norm Evolution

Weight norms are a key diagnostic in grokking (rfm-grokking style).  During
memorisation norms grow; once weight decay forces them down, the network
transitions to a generalising solution — the grokking phase transition.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for label, h, c in configs:
    ax.plot(h['epoch'], h['cb/weight_norm'], label=label, color=c, alpha=0.8)

ax.set(xlabel='Epoch', ylabel='Total Weight L2 Norm',
       title='Weight Norm Evolution (rfm-grokking style diagnostic)')
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('weight_norm_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: weight_norm_comparison.png')

### 5d. Gradient Norm Evolution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.semilogy(baseline_hist['epoch'], baseline_hist['cb/gradient_norm'],
            label='Baseline Transformer', color='tab:blue', alpha=0.8)
ax.semilogy(gf_hist['epoch'], gf_hist['cb/gradient_norm'],
            label='Grokfast-EMA Transformer', color='tab:orange', alpha=0.8)

ax.set(xlabel='Epoch', ylabel='Gradient Norm (log)',
       title='Gradient Norm Evolution')
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('gradient_norm_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: gradient_norm_comparison.png')

### 5e. Summary Table

In [ ]:
def find_grok_epoch(hist, threshold=0.95):
    """Find the first epoch where val accuracy exceeds the threshold."""
    for i, acc in enumerate(hist['val_acc']):
        if acc >= threshold:
            return hist['epoch'][i]
    return None

results = [
    ('Baseline Transformer', baseline_hist),
    ('Grokfast-EMA Transformer', gf_hist),
    ('MLP Baseline', mlp_hist),
]

print(f"{'Experiment':<30} {'Final Train':>12} {'Final Val':>12} {'Grok@95%':>12} {'Time(s)':>10}")
print('-' * 80)
for name, h in results:
    grok_ep = find_grok_epoch(h, 0.95)
    grok_str = str(grok_ep) if grok_ep else 'Not reached'
    print(f"{name:<30} {h['train_acc'][-1]:>12.4f} {h['val_acc'][-1]:>12.4f} {grok_str:>12} {h['elapsed_sec'][-1]:>10.1f}")

---

## 6. Extensibility — Adding a Custom Metric / Hypothesis

The framework has **four extension points**:

| Extension | How | Decorator |
|-----------|-----|----------|
| New **metric / hypothesis** | Subclass `MetricCallback`, implement `on_epoch_end` | `@register_metric("name")` |
| New **operation / dataset** | Write a function `(a, b, p) -> int` | `@register_operation("name")` |
| New **model architecture** | Subclass `nn.Module` with `forward(x) -> logits` | `@register_model("name")` |
| New **optimizer mod** | Create a class with `.apply()` that modifies `.grad` | Pass to `Trainer(grokfast=...)` |

### Example: Tracking "Effective Rank" of weight matrices

The effective rank (based on Shannon entropy of normalised singular values)
measures how many directions in weight space are actually being used.

In [ ]:
@register_metric("effective_rank")
class EffectiveRankCallback(MetricCallback):
    """Tracks the effective rank of the unembed weight matrix."""

    def __init__(self, layer_filter: str = "unembed"):
        super().__init__()
        self.layer_filter = layer_filter

    def on_epoch_end(self, model, epoch, **kwargs):
        for name, param in model.named_parameters():
            if self.layer_filter in name and param.ndim == 2:
                svs = torch.linalg.svdvals(param.data)
                p = svs / svs.sum()
                p = p[p > 1e-10]
                entropy = -(p * p.log()).sum().item()
                self._log(f"eff_rank/{name}", float(np.exp(entropy)))

print(f"Registered! All metrics: {get_available_metrics()}")

In [ ]:
torch.manual_seed(SEED)
er_model = create_model(
    "transformer", p=PRIME, d_model=128, n_heads=4, d_mlp=512, n_layers=1
)
er_opt = torch.optim.AdamW(
    er_model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY, betas=(0.9, 0.98)
)
er_cb = EffectiveRankCallback(layer_filter="unembed")

er_trainer = Trainer(
    model=er_model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=er_opt,
    callbacks=[er_cb],
    device=DEVICE,
    log_interval=LOG_INTERVAL,
)

print("Training with custom effective-rank metric...")
er_hist = er_trainer.train(epochs=EPOCHS)

In [ ]:
er_keys = [k for k in er_hist if k.startswith('cb/eff_rank/')]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

ax1.plot(er_hist['epoch'], er_hist['val_acc'],   color='tab:blue', label='Val Accuracy')
ax1.plot(er_hist['epoch'], er_hist['train_acc'], color='tab:blue', alpha=0.3,
         linestyle='--', label='Train Accuracy')
ax1.set_ylabel('Accuracy')
ax1.set_title('Custom Hypothesis: Does effective rank change during grokking?')
ax1.legend(); ax1.grid(True, alpha=0.3)

for k in er_keys:
    ax2.plot(er_hist['epoch'], er_hist[k], label=k.split('/')[-1], alpha=0.8)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Effective Rank')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('effective_rank_hypothesis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: effective_rank_hypothesis.png')

## 7. Extensibility — Adding a Custom Operation

In [ ]:
@register_operation("x3_plus_y3")
def _op_x3_plus_y3(a, b, p):
    """a^3 + b^3 mod p"""
    return (a**3 + b**3) % p

print(f"Available operations: {get_available_operations()}")

ds_custom = ModularArithmeticDataset("x3_plus_y3", p=PRIME, train_fraction=0.3)
print(ds_custom)

---

## Quick Reference — Adding Your Own Hypothesis

```python
# 1. Define your metric
@register_metric("my_hypothesis")
class MyHypothesisCallback(MetricCallback):
    def on_epoch_end(self, model, epoch, **kwargs):
        value = ...  # compute something from model
        self._log("my_value", value)

# 2. Add it to a Trainer
trainer = Trainer(
    model=..., train_loader=..., val_loader=..., optimizer=...,
    callbacks=[MyHypothesisCallback()],
)
hist = trainer.train(epochs=2000)

# 3. Plot it
plt.plot(hist['epoch'], hist['cb/my_value'])
```